<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/33_rag_guardrails/rag_guardrails.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install numpy
!pip install pandas

In [ ]:
import numpy as np
import pandas as pd

In [17]:
documents = [
    "Elon Musk founded SpaceX.",
    "SpaceX works on rockets and space exploration.",
    "Tesla builds electric cars.",
    "Elon Musk is CEO of Tesla."
]

df = pd.DataFrame({"text": documents})

In [ ]:
def retrieve(query):
    query_words = query.lower().split()
    scores = []

    for text in df["text"]:
        text_lower = text.lower()
        score = sum(word in text_lower for word in query_words)
        scores.append(score)

    scores = np.array(scores)
    top_idx = np.argmax(scores)

    return df.iloc[top_idx]["text"], scores[top_idx]

In [ ]:
def extract_answer(context):
    context_lower = context.lower()

    if "founded" in context_lower:
        return "Elon Musk"
    elif "spacex" in context_lower and "rocket" in context_lower:
        return "SpaceX"
    elif "tesla" in context_lower:
        return "Tesla"
    else:
        return "Answer not found"

In [ ]:
def apply_guardrails(query, context, score):
    query_lower = query.lower()

    # 1. Low confidence
    if score == 0:
        return "I don't have enough information to answer that."

    # 2. Domain check (ALLOW only known domain)
    allowed_keywords = ["spacex", "tesla", "elon", "rocket", "car", "founded", "ceo"]

    if not any(word in query_lower for word in allowed_keywords):
        return "This question is outside my knowledge domain."

    return None

In [ ]:
def rag_with_guardrails(query):
    context, score = retrieve(query)

    # Apply guardrails FIRST
    guardrail_response = apply_guardrails(query, context, score)

    if guardrail_response:
        print("\nGuardrail Activated 🚫")
        print("Response:", guardrail_response)
        return

    # If safe → answer
    answer = extract_answer(context)

    print("\nQuery:", query)
    print("Context:", context)
    print("Answer:", answer)


In [18]:
rag_with_guardrails("Which company works on rockets?")
rag_with_guardrails("Who founded Tesla?")
rag_with_guardrails("What is the weather today?")


Query: Which company works on rockets?
Context: SpaceX works on rockets and space exploration.
Answer: SpaceX

Query: Who founded Tesla?
Context: Elon Musk founded SpaceX.
Answer: Elon Musk

Guardrail Activated 🚫
Response: This question is outside my knowledge domain.
